# JSONを識別子.metadata.jsonとして保存

In [1]:
from pathlib import Path

# Notebookと同じ階層を基準にする
BASE_DIR = Path.cwd()

# Markdownフォルダ
MARKDOWN_DIR = BASE_DIR / "markdown"

# ファイル名変更処理
def rename_metadata_files(folder: Path, dry_run=True):

    json_files = list(folder.glob("*.json"))

    for json_file in json_files:

        # 対応するMarkdownファイルを取得
        md_file = json_file.with_suffix(".md")

        if not md_file.exists():
            print(f"スキップ（対応するmdなし）: {json_file.name}")
            continue

        # Bedrock用のメタデータファイル名
        new_name = folder / f"{md_file.name}.metadata.json"

        # 既に存在する場合は上書きしない
        if new_name.exists():
            print(f"スキップ（既に存在）: {new_name.name}")
            continue

        if dry_run:
            print(f"[確認] {json_file.name} → {new_name.name}")
        else:
            json_file.rename(new_name)
            print(f"[変更] {json_file.name} → {new_name.name}")


# まずは変更内容を確認
rename_metadata_files(MARKDOWN_DIR, dry_run=True)


[確認] 2026-10-17.json → 2026-10-17.md.metadata.json
[確認] 2026-10-16.json → 2026-10-16.md.metadata.json
[確認] 2026-10-15.json → 2026-10-15.md.metadata.json
[確認] 2026-10-19.json → 2026-10-19.md.metadata.json
[確認] 2026-10-18.json → 2026-10-18.md.metadata.json


In [3]:
# 実際に変更を行う。
rename_metadata_files(MARKDOWN_DIR, dry_run=False)

スキップ（対応するmdなし）: 2026-10-15.md.metadata.json
スキップ（対応するmdなし）: 2026-10-18.md.metadata.json
スキップ（対応するmdなし）: 2026-10-17.md.metadata.json
スキップ（対応するmdなし）: 2026-10-19.md.metadata.json
スキップ（対応するmdなし）: 2026-10-16.md.metadata.json


# kb の metadata の形式が違ったため、修正を行って実施する。

In [4]:
import json
import shutil
from pathlib import Path

# Notebookと同じ階層を取得
BASE_DIR = Path.cwd()

# Markdownフォルダ
MARKDOWN_DIR = BASE_DIR / "markdown"

# バックアップフォルダ
BACKUP_DIR = BASE_DIR / "metadata_backup"

# フォルダの存在確認
assert MARKDOWN_DIR.exists(), f"フォルダがありません: {MARKDOWN_DIR}"

print("Markdownフォルダ:", MARKDOWN_DIR)
print("Backupフォルダ:", BACKUP_DIR)

Markdownフォルダ: /Users/black/Downloads/Code/Available/generative_ai/azure/training/22_RAG/bedrock/markdown
Backupフォルダ: /Users/black/Downloads/Code/Available/generative_ai/azure/training/22_RAG/bedrock/metadata_backup


In [5]:
# メタデータファイルを取得
metadata_files = sorted(
    MARKDOWN_DIR.glob("*.md.metadata.json")
)

print(f"対象ファイル数: {len(metadata_files)}")

for file_path in metadata_files:
    print(file_path.name)


対象ファイル数: 5
2026-10-15.md.metadata.json
2026-10-16.md.metadata.json
2026-10-17.md.metadata.json
2026-10-18.md.metadata.json
2026-10-19.md.metadata.json


In [6]:

def convert_metadata_files(folder, backup_dir, dry_run=True):

    files = sorted(folder.glob("*.md.metadata.json"))

    converted = 0
    skipped = 0
    errors = 0

    for file_path in files:

        try:
            # JSON読み込み
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            # JSON形式の確認
            if not isinstance(data, dict):
                raise ValueError("JSONのルートが辞書形式ではありません")

            # 対応するMarkdownファイルの確認
            md_file = folder / file_path.name.replace(
                ".metadata.json", ""
            )

            if not md_file.exists():
                print(f"[SKIP] Markdownなし: {file_path.name}")
                skipped += 1
                continue

            # 既にBedrock形式ならスキップ
            if "metadataAttributes" in data:
                print(f"[SKIP] 変換済み: {file_path.name}")
                skipped += 1
                continue

            # 変換後のデータ
            converted_data = {
                "metadataAttributes": data
            }

            # AWSの10KB制限を確認
            json_bytes = json.dumps(
                converted_data,
                ensure_ascii=False,
                indent=2
            ).encode("utf-8")

            if len(json_bytes) > 10 * 1024:
                raise ValueError("メタデータが10KBを超えています")

            # 変更内容の確認
            if dry_run:
                print(f"[確認] {file_path.name}")

            else:
                # バックアップ
                backup_dir.mkdir(parents=True, exist_ok=True)
                shutil.copy2(
                    file_path,
                    backup_dir / file_path.name
                )

                # JSONを書き換え
                with open(file_path, "w", encoding="utf-8") as f:
                    json.dump(
                        converted_data,
                        f,
                        ensure_ascii=False,
                        indent=2
                    )

                print(f"[変換完了] {file_path.name}")

            converted += 1

        except Exception as e:
            print(f"[ERROR] {file_path.name}: {e}")
            errors += 1

    print("\n===== 処理結果 =====")
    print(f"対象ファイル数: {len(files)}")
    print(f"変換対象・完了: {converted}")
    print(f"スキップ: {skipped}")
    print(f"エラー: {errors}")


convert_metadata_files(
    MARKDOWN_DIR,
    BACKUP_DIR,
    dry_run=True
)

[確認] 2026-10-15.md.metadata.json
[確認] 2026-10-16.md.metadata.json
[確認] 2026-10-17.md.metadata.json
[確認] 2026-10-18.md.metadata.json
[確認] 2026-10-19.md.metadata.json

===== 処理結果 =====
対象ファイル数: 5
変換対象・完了: 5
スキップ: 0
エラー: 0


In [7]:
convert_metadata_files(
    MARKDOWN_DIR,
    BACKUP_DIR,
    dry_run=False
)

[変換完了] 2026-10-15.md.metadata.json
[変換完了] 2026-10-16.md.metadata.json
[変換完了] 2026-10-17.md.metadata.json
[変換完了] 2026-10-18.md.metadata.json
[変換完了] 2026-10-19.md.metadata.json

===== 処理結果 =====
対象ファイル数: 5
変換対象・完了: 5
スキップ: 0
エラー: 0


### 変換されたか確認

In [8]:
# 先頭のメタデータファイルを確認
metadata_files = sorted(
    MARKDOWN_DIR.glob("*.md.metadata.json")
)

if metadata_files:

    target_file = metadata_files[0]

    with open(target_file, "r", encoding="utf-8") as f:
        result = json.load(f)

    print("ファイル:", target_file.name)
    print(json.dumps(
        result,
        ensure_ascii=False,
        indent=2
    ))

    # 変換確認
    assert "metadataAttributes" in result

    print("\nメタデータ形式の確認: OK")


ファイル: 2026-10-15.md.metadata.json
{
  "metadataAttributes": {
    "date": "2026-10-15",
    "date_confidence": "high",
    "category": "クラウド",
    "category_confidence": "middle"
  }
}

メタデータ形式の確認: OK
